# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [1]:
import gradio as gr
from datetime import datetime
from openai import OpenAI

# connect to the local Ollama server
ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# models that will appear in the dropdown
MODELS = ["llama3.2"]

# system prompt: gives the model its expertise
system_prompt = """You are an expert technical tutor.
You answer programming and computer science questions clearly and correctly.
Use short code examples when they help, and keep explanations beginner-friendly."""

# the tool: a normal Python function the model can ask us to run
def get_current_time():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# description of the tool, so the model knows it exists and when to use it
tools = [{
    "type": "function",
    "function": {
        "name": "get_current_time",
        "description": "Get the current date and time on the user's computer",
        "parameters": {"type": "object", "properties": {}, "required": []},
    },
}]

def chat(message, history, model):
    # keep only role and content from Gradio's history
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]

    # first call (no streaming): check whether the model wants to use the tool
    try:
        response = ollama.chat.completions.create(model=model, messages=messages, tools=tools)
        first = response.choices[0].message
        if first.tool_calls:
            messages.append(first)
            for call in first.tool_calls:
                messages.append({
                    "role": "tool",
                    "tool_call_id": call.id,
                    "content": get_current_time(),
                })
    except Exception:
        # some models don't support tools, so just carry on without them
        pass

    # final call: stream the answer piece by piece
    stream = ollama.chat.completions.create(model=model, messages=messages, stream=True)
    reply = ""
    for chunk in stream:
        reply += chunk.choices[0].delta.content or ""
        yield reply

# the Gradio UI, with a dropdown for switching models
gr.ChatInterface(
    fn=chat,
    type="messages",
    additional_inputs=[gr.Dropdown(MODELS, value=MODELS[0], label="Model")],
).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
